# FLOW — Large-Scale Vocabulary Growth on Kaggle

**Geometric Causal Architecture** — weight-free, token-free reasoning.  
Knowledge stored as shape in a 104D Riemannian manifold.

This notebook feeds **multiple real-world corpora** through the Phase 7 VocabularyBuilder pipeline  
to grow the expression vocabulary from ~100 entries to **10,000–20,000** entries.

### Corpus mix — grounded in agentic capabilities

| Dataset | Domain | Why it matters |
|---|---|---|
| Simple English Wikipedia | General knowledge | Broad concept coverage, clean prose |
| OpenAssistant (oasst1) | Agent assistance / dialogue | Conversational patterns, Q&A structure |
| Alpaca (cleaned) | Instruction following | Imperative/procedural language |
| Glaive Function Calling v2 | Tool calling / function use | Structured action language, parameters |
| SlimOrca | Reasoning chains | Step-by-step explanation patterns |

| Metric | Before | Target |
|---|---|---|
| Vocabulary entries | ~100 | 10K–20K |
| Manifold concepts | ~137 | 5K–15K |
| Output fluency | Proof-of-concept | Coherent |

**Hardware:** Kaggle CPU (4 cores, 30 GB RAM)  
**Estimated runtime:** 15–30 minutes  
**Output artifacts:** `vocabulary_large.npz`, `manifold_large.npz`

## 1. Setup — Clone repo & install dependencies

In [ ]:
# Clone the FLOW repository
!git clone https://github.com/Unseengap/FLOW

# Install dependencies (pure math — no ML frameworks)
!pip install numpy scipy networkx datasets -q

# GPU acceleration (optional — graceful fallback to CPU)
!pip install cupy-cuda12x faiss-gpu -q 2>/dev/null || echo "GPU packages not available, using CPU fallback"

# Verify
import os
assert os.path.isdir("FLOW/src"), "FLOW repo not found"
print("✓ FLOW repository cloned successfully")

# GPU status
try:
    import cupy as cp
    print(f"✓ cupy detected — GPU: {cp.cuda.runtime.getDeviceProperties(0)['name'].decode()}")
    print(f"  VRAM: {cp.cuda.runtime.memGetInfo()[1] / 1024**3:.1f} GB")
except Exception:
    print("⊘ cupy not available — batch ops will use numpy (still fast)")

try:
    import faiss
    if faiss.get_num_gpus() > 0:
        print(f"✓ FAISS-GPU detected — {faiss.get_num_gpus()} GPU(s)")
    else:
        print("✓ FAISS-CPU detected (no GPU index)")
except ImportError:
    print("⊘ FAISS not available — using linear scan fallback")

In [ ]:
# Add FLOW to Python path
import sys
sys.path.insert(0, "FLOW")

# Verify imports
from src.phase5 import GEOPipeline, PipelineResult
from src.vocabulary import VocabularyBuilder, VocabularyStore
from src.persistence import ManifoldSnapshot
from src.phase3.annealing_engine.experience import Experience
from src.phase1.expression.matcher import ExpressionEntry
import numpy as np
import time

print("✓ All FLOW imports successful")
print(f"  numpy {np.__version__}")

## 2. Build the GEOPipeline

In [ ]:
# Build the full C1-C7 pipeline
# T0=1.0 : high initial temperature for flexible placement
# lambda_=0.005 : slower cooling for large-scale ingestion
# T_floor=0.03  : lower floor to allow crystallisation over time

pipeline = GEOPipeline(
    T0=1.0,
    lambda_=0.005,
    T_floor=0.03,
    flow_seed=42,
)

print(f"✓ Pipeline built")
print(f"  Manifold dimension : {pipeline.manifold.DIM}")
print(f"  Seed concepts      : {pipeline.n_concepts}")
print(f"  Temperature        : {pipeline.temperature:.4f}")

## 3. Load corpora — multi-domain mix

We feed five complementary datasets that ground FLOW in:
- **General knowledge** — Wikipedia (broad concept coverage)
- **Agent assistance** — OpenAssistant conversations (Q&A, dialogue)
- **Instruction following** — Stanford Alpaca (imperative/procedural text)
- **Tool calling** — Glaive Function Calling (structured actions, parameters)
- **Reasoning chains** — SlimOrca (step-by-step explanations)

Each dataset contributes different linguistic patterns to the manifold geometry.

In [ ]:
from datasets import load_dataset

all_texts = []   # collect (text, source_tag) pairs for unified feeding

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Dataset limits — 10K each for a fast, capable proof of concept
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
N_PER_DATASET = 10_000

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3A. Wikipedia — Simple English (general knowledge)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("Loading Simple English Wikipedia...")
ds_wiki = load_dataset("wikimedia/wikipedia", "20231101.simple", split="train", trust_remote_code=True)
for i, article in enumerate(ds_wiki):
    if i >= N_PER_DATASET:
        break
    text = article.get("text", "")
    if text and len(text) > 50:
        all_texts.append((text, "wikipedia"))
print(f"  ✓ Wikipedia: {len([t for t in all_texts if t[1]=='wikipedia']):,} articles")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3B. OpenAssistant (oasst1) — agent assistance / dialogue
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("Loading OpenAssistant (oasst1)...")
ds_oasst = load_dataset("OpenAssistant/oasst1", split="train", trust_remote_code=True)
for i, row in enumerate(ds_oasst):
    if i >= N_PER_DATASET:
        break
    text = row.get("text", "")
    if text and len(text) > 30:
        all_texts.append((text, "openassistant"))
print(f"  ✓ OpenAssistant: {len([t for t in all_texts if t[1]=='openassistant']):,} messages")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3C. Alpaca (cleaned) — instruction following
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("Loading Alpaca (cleaned)...")
ds_alpaca = load_dataset("yahma/alpaca-cleaned", split="train", trust_remote_code=True)
for i, row in enumerate(ds_alpaca):
    if i >= N_PER_DATASET:
        break
    # Concatenate instruction + input + output for full context
    parts = []
    if row.get("instruction"):
        parts.append(row["instruction"])
    if row.get("input"):
        parts.append(row["input"])
    if row.get("output"):
        parts.append(row["output"])
    text = " ".join(parts)
    if text and len(text) > 30:
        all_texts.append((text, "alpaca"))
print(f"  ✓ Alpaca: {len([t for t in all_texts if t[1]=='alpaca']):,} instructions")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3D. Glaive Function Calling v2 — tool calling / function use
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("Loading Glaive Function Calling v2...")
ds_glaive = load_dataset("glaiveai/glaive-function-calling-v2", split="train", trust_remote_code=True)
for i, row in enumerate(ds_glaive):
    if i >= N_PER_DATASET:
        break
    # Each row has system prompt + chat — extract all text
    text_parts = []
    if row.get("system"):
        text_parts.append(row["system"])
    if row.get("chat"):
        text_parts.append(row["chat"])
    text = " ".join(text_parts)
    if text and len(text) > 30:
        all_texts.append((text, "glaive_functions"))
print(f"  ✓ Glaive Functions: {len([t for t in all_texts if t[1]=='glaive_functions']):,} examples")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3E. SlimOrca — reasoning chains
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("Loading SlimOrca...")
ds_orca = load_dataset("Open-Orca/SlimOrca", split="train", trust_remote_code=True)
for i, row in enumerate(ds_orca):
    if i >= N_PER_DATASET:
        break
    # SlimOrca uses 'conversations' list with role/value dicts
    convos = row.get("conversations", [])
    text = " ".join(turn.get("value", "") for turn in convos if isinstance(turn, dict))
    if text and len(text) > 30:
        all_texts.append((text, "slimorca"))
print(f"  ✓ SlimOrca: {len([t for t in all_texts if t[1]=='slimorca']):,} reasoning chains")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Summary
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from collections import Counter
source_counts = Counter(t[1] for t in all_texts)
print(f"\n{'='*60}")
print(f"Total texts loaded: {len(all_texts):,}")
for src, cnt in source_counts.most_common():
    print(f"  {src:20s} : {cnt:>8,}")
print(f"{'='*60}")

## 4. Configure VocabularyBuilder & feed all corpora

In [ ]:
# ── Configuration ──────────────────────────────────────────────
# Vocabulary ceiling — controls how many unique words we keep.
# 10K is a fast, capable proof of concept (~15-30 min total).
V_MAX = 10_000          # target vocabulary size
MIN_COUNT = 5            # minimum frequency for a word to be kept
WINDOW_SIZE = 5          # co-occurrence sliding window
N_CONTRAST_PASSES = 2    # C4 refinement passes over PMI matrix
TAU_SAME = 1.0           # PMI threshold for SAME judgment
TAU_DIFF = -0.5          # PMI threshold for DIFFERENT judgment
BATCH_SIZE = 512         # contrast judgment batch size

print(f"Configuration:")
print(f"  V_MAX            = {V_MAX:,}")
print(f"  MIN_COUNT        = {MIN_COUNT}")
print(f"  WINDOW_SIZE      = {WINDOW_SIZE}")
print(f"  N_CONTRAST_PASSES= {N_CONTRAST_PASSES}")
print(f"  Total texts      = {len(all_texts):,}")

In [ ]:
# Build the VocabularyBuilder
builder = VocabularyBuilder(
    manifold=pipeline.manifold,
    annealing_engine=pipeline._annealing,
    contrast_engine=pipeline._contrast_engine,
    window_size=WINDOW_SIZE,
    min_count=MIN_COUNT,
    v_max=V_MAX,
    tau_same=TAU_SAME,
    tau_diff=TAU_DIFF,
    batch_size=BATCH_SIZE,
    n_contrast_passes=N_CONTRAST_PASSES,
)

print("✓ VocabularyBuilder created")

In [ ]:
# ── Feed all corpora ──────────────────────────────────────────────
# Shuffled to interleave domains — prevents the manifold from
# crystallising on one domain before seeing others.

import random
random.seed(42)
random.shuffle(all_texts)

t0 = time.time()
REPORT_EVERY = 25_000
n_total = len(all_texts)

for i, (text, source) in enumerate(all_texts):
    builder.feed(text)
    
    if (i + 1) % REPORT_EVERY == 0:
        elapsed = time.time() - t0
        rate = (i + 1) / elapsed
        print(f"  [{i+1:>8,} / {n_total:,}]  "
              f"tokens: {builder.n_tokens_fed:>12,}  "
              f"({rate:.0f} texts/sec, {elapsed:.1f}s elapsed)")

elapsed = time.time() - t0
print(f"\n✓ All corpora ingested")
print(f"  Texts processed    : {n_total:,}")
print(f"  Total tokens       : {builder.n_tokens_fed:,}")
print(f"  Time               : {elapsed:.1f}s ({elapsed/60:.1f} min)")

## 5. Build vocabulary — broken into stages with full logging

The build pipeline runs 6 steps, each in its own cell so you can see
progress, catch errors, and know the system isn't hanging:

1. **5A** — Build PMI matrix from co-occurrence counts
2. **5B** — Place all vocabulary words on M(t) via C3 Annealing (GPU-accelerated)
3. **5C** — Contrast refinement via C4 (SAME/DIFFERENT from PMI)
4. **5D** — Calibrate phrase radius
5. **5E** — Build ExpressionEntry objects (Level 1 + 2 + 3)
6. **5F** — Save to `.npz`

In [ ]:
# ── 5A: Build PMI matrix from co-occurrence counts ──────────────
# This converts all the raw text counts into pairwise PMI scores.
# Should be fast (~10–60s) — it's pure arithmetic on count dicts.

import traceback

OUTPUT_DIR = "/kaggle/working"    # Kaggle output directory
VOCAB_PATH = f"{OUTPUT_DIR}/vocabulary_large.npz"

print("═" * 70)
print("  STEP 5A — Build PMI matrix")
print("═" * 70)
t0 = time.time()

try:
    matrix = builder._counter.build()
    builder._matrix = matrix
    elapsed = time.time() - t0

    print(f"  ✓ PMI matrix built in {elapsed:.1f}s")
    print(f"  Vocabulary size    : {len(matrix.vocabulary):,}")
    print(f"  PMI pairs          : {len(matrix._pmi):,}")
    print(f"  Directed PMI pairs : {len(matrix._dpmi):,}")
    print(f"  PMI max            : {matrix.pmi_max():.4f}")

    # Show top-10 strongest PMI pairs
    top_pairs = matrix.pairs_above_threshold(1.0, -99)[:10]
    print(f"\n  Top 10 PMI pairs:")
    for w1, w2, pmi in top_pairs:
        print(f"    PMI={pmi:>7.3f}  {w1} ↔ {w2}")

    # Count how many pairs will need contrast judgments
    same_pairs = [p for p in matrix.pairs_above_threshold(builder._scheduler.tau_same, -99) if p[2] > builder._scheduler.tau_same]
    diff_pairs = [p for p in matrix.pairs_above_threshold(99, builder._scheduler.tau_diff) if p[2] < builder._scheduler.tau_diff]
    causal_pairs = matrix.directed_pairs_above_delta(builder._scheduler.delta_causal)
    print(f"\n  Contrast workload preview:")
    print(f"    SAME pairs (PMI > {builder._scheduler.tau_same})  : {len(same_pairs):,}")
    print(f"    DIFF pairs (PMI < {builder._scheduler.tau_diff}) : {len(diff_pairs):,}")
    print(f"    Causal pairs (δ > {builder._scheduler.delta_causal})   : {len(causal_pairs):,}")
    print(f"    Total judgments × {builder.n_contrast_passes} passes ≈ {(len(same_pairs) + len(diff_pairs)) * builder.n_contrast_passes:,}")

except Exception as e:
    print(f"  ✗ FAILED: {e}")
    traceback.print_exc()
    raise

print(f"\n{'═' * 70}")
print(f"  5A COMPLETE — {elapsed:.1f}s")
print(f"{'═' * 70}")

In [ ]:
# ── 5B: Place vocabulary words on M(t) via fast path ─────────────
# Each word gets a 104D structural vector placed directly on M(t).
# OPTIMIZED:
#   - place_fast() skips per-word density/curvature recomputation
#   - KDTree rebuild throttled (rebuild every 50 writes, not every write)
#   - flush_batch() at end does one-pass density + tree rebuild
#   - GPU path: structural vectors computed on GPU if cupy available
# Expected: ~60-120s for 50K words (was 2.4 hours before optimization)

from src.vocabulary.word_placer import WordPlacer, batch_structural_vectors_gpu

print("═" * 70)
print("  STEP 5B — Place vocabulary words on M(t)")
print("═" * 70)

vocab = matrix.vocabulary
freq_ranks = list(range(1, len(vocab) + 1))
n_words = len(vocab)
print(f"  Words to place     : {n_words:,}")
print(f"  Manifold before    : {pipeline.n_concepts:,} concepts")
print(f"  Temperature        : {pipeline.temperature:.6f}")

# Progress callback — shows we're not hanging
def placement_progress(i, total, label):
    elapsed = time.time() - t_place
    rate = i / elapsed if elapsed > 0 else 0
    pct = 100 * i / total
    print(f"    [{i:>6,}/{total:,}] {pct:5.1f}%  "
          f"{rate:,.0f} words/sec  "
          f"T={pipeline.temperature:.6f}  "
          f"last: {label}")

t_place = time.time()
try:
    # Use GPU path if cupy is available, else optimized CPU batch
    try:
        import cupy as _cp
        print(f"  → Using GPU-accelerated structural vectors (cupy)")
        placed = builder._placer.place_batch_gpu(vocab, freq_ranks,
                                                   progress_callback=placement_progress)
    except ImportError:
        print(f"  → Using optimized CPU batch placement")
        placed = builder._placer.place_batch(vocab, freq_ranks,
                                              progress_callback=placement_progress)

    builder._n_words_placed = len(vocab)
    elapsed_place = time.time() - t_place

    print(f"\n  ✓ Word placement complete")
    print(f"  Words placed       : {len(placed):,}")
    print(f"  Manifold after     : {pipeline.n_concepts:,} concepts")
    print(f"  Time               : {elapsed_place:.1f}s ({elapsed_place/60:.1f} min)")
    print(f"  Rate               : {len(placed)/elapsed_place:,.0f} words/sec")
    print(f"  Temperature now    : {pipeline.temperature:.6f}")

    # Sanity check — spot-check a few placements
    for w in vocab[:3]:
        lbl = f"vocab::{w}"
        try:
            pos = pipeline.manifold.position(lbl)
            d = pipeline.manifold.density(pos)
            print(f"    ✓ {lbl:30s}  ρ={d:.4f}  ‖P‖={np.linalg.norm(pos):.4f}")
        except (KeyError, ValueError) as e:
            print(f"    ✗ {lbl} — NOT FOUND: {e}")

except Exception as e:
    elapsed_place = time.time() - t_place
    print(f"\n  ✗ FAILED after {elapsed_place:.1f}s: {e}")
    traceback.print_exc()
    raise

print(f"\n{'═' * 70}")
print(f"  5B COMPLETE — {elapsed_place:.1f}s ({elapsed_place/60:.1f} min)")
print(f"{'═' * 70}")

In [ ]:
# ── 5C: Contrast refinement — C4 SAME/DIFFERENT judgments ─────────
# Iterates all PMI pairs N_CONTRAST_PASSES times.
# OPTIMIZED:
#   - judge_fast() skips per-pair density updates + persistence tracking
#   - KDTree rebuild throttled (every 50 writes, not every deform)
#   - Neighbor cap (max 64) prevents O(n) in dense regions
#   - Tree rebuild forced between passes for spatial accuracy
#   - Pre-built label set for O(1) membership checks
# Expected: ~5-15 min for 3 passes (was estimated 6-12+ hours before)

print("═" * 70)
print("  STEP 5C — Contrast refinement (C4)")
print("═" * 70)
print(f"  Passes             : {builder.n_contrast_passes}")
print(f"  Batch size         : {builder._scheduler.batch_size}")
print(f"  τ_same             : {builder._scheduler.tau_same}")
print(f"  τ_diff             : {builder._scheduler.tau_diff}")
print(f"  Manifold labels    : {len(pipeline.manifold.labels):,}")

# Contrast progress callback
_contrast_last_report = [time.time()]
def contrast_progress(n_applied, n_total, phase):
    now = time.time()
    if now - _contrast_last_report[0] < 10.0 and "done" not in phase:
        return  # throttle to every 10s
    _contrast_last_report[0] = now
    elapsed = now - t_contrast
    if "pass" in phase and "done" in phase:
        print(f"    ── {phase}: {n_applied:,} judgments applied ({elapsed:.0f}s elapsed)")
    elif "done" in phase:
        print(f"    ── {phase}: {n_applied:,} judgments so far ({elapsed:.0f}s)")
    elif n_total > 0:
        pct = min(100, 100 * n_applied / max(n_total, 1))
        rate = n_applied / elapsed if elapsed > 0 else 0
        print(f"    [{n_applied:>8,} / ~{n_total:,}] {pct:5.1f}%  "
              f"{rate:,.0f} judg/sec ({elapsed:.0f}s)")

t_contrast = time.time()
try:
    n_judgments = builder._scheduler.run_passes(
        matrix,
        n_passes=builder.n_contrast_passes,
        progress_callback=contrast_progress,
    )
    builder._n_judgments = n_judgments
    elapsed_contrast = time.time() - t_contrast

    print(f"\n  ✓ Contrast refinement complete")
    print(f"  Total judgments    : {n_judgments:,}")
    print(f"  Time               : {elapsed_contrast:.1f}s ({elapsed_contrast/60:.1f} min)")
    print(f"  Rate               : {n_judgments/elapsed_contrast:,.0f} judgments/sec")

    # Spot-check: did geometry actually change?
    if len(vocab) >= 2:
        w1, w2 = vocab[0], vocab[1]
        d = float(np.linalg.norm(
            pipeline.manifold.position(f"vocab::{w1}") -
            pipeline.manifold.position(f"vocab::{w2}")
        ))
        print(f"  Sample dist({w1}, {w2}) = {d:.4f}")

except Exception as e:
    elapsed_contrast = time.time() - t_contrast
    print(f"\n  ✗ FAILED after {elapsed_contrast:.1f}s: {e}")
    traceback.print_exc()
    raise

print(f"\n{'═' * 70}")
print(f"  5C COMPLETE — {elapsed_contrast:.1f}s ({elapsed_contrast/60:.1f} min)")
print(f"{'═' * 70}")

In [ ]:
# ── 5D: Calibrate phrase radius ───────────────────────────────────
# Computes the mean pairwise distance within the top-density clusters.
# Fast (~1-5s). Determines how close words must be to form phrases.

from src.vocabulary.template_builder import TemplateBuilder

print("═" * 70)
print("  STEP 5D — Calibrate phrase radius")
print("═" * 70)

t_calib = time.time()
try:
    tb = TemplateBuilder(pipeline.manifold)
    radius = tb.calibrate_phrase_radius()
    elapsed_calib = time.time() - t_calib

    print(f"  ✓ Phrase radius calibrated")
    print(f"  Radius             : {radius:.4f}")
    print(f"  Time               : {elapsed_calib:.2f}s")

    # Show how many potential phrase pairs exist at this radius
    vocab_labels = [l for l in pipeline.manifold._points if l.startswith("vocab::")]
    print(f"  Vocab on manifold  : {len(vocab_labels):,}")

except Exception as e:
    elapsed_calib = time.time() - t_calib
    print(f"\n  ✗ FAILED after {elapsed_calib:.2f}s: {e}")
    traceback.print_exc()
    raise

print(f"\n{'═' * 70}")
print(f"  5D COMPLETE — {elapsed_calib:.2f}s")
print(f"{'═' * 70}")

In [ ]:
# ── 5E: Build ExpressionEntry objects (Level 1 + 2 + 3) ───────────
# Level 1: Single words (~50K)     — batch vectorized, GPU-accelerated
# Level 2: Short phrases (~35K)    — cKDTree batch kNN
# Level 3: Sentence frames (~16)   — template-based
# OPTIMIZED: batch hedging via cKDTree replaces 50K individual kNN calls.

print("═" * 70)
print("  STEP 5E — Build ExpressionEntry objects")
print("═" * 70)

t_build = time.time()
try:
    # Level 1 — single words
    print("  Building Level 1 (single words)...")
    t_l1 = time.time()
    l1_entries = tb._build_level1()
    elapsed_l1 = time.time() - t_l1
    print(f"    ✓ Level 1: {len(l1_entries):,} entries in {elapsed_l1:.1f}s")
    if l1_entries:
        sample = l1_entries[0]
        print(f"    Sample: '{sample.text}' register={sample.register} "
              f"causal={sample.causal_strength:.3f} hedging={sample.hedging}")

    # Level 2 — short phrases
    print("  Building Level 2 (short phrases)...")
    t_l2 = time.time()
    l2_entries = tb._build_level2(matrix)
    elapsed_l2 = time.time() - t_l2
    print(f"    ✓ Level 2: {len(l2_entries):,} entries in {elapsed_l2:.1f}s")
    if l2_entries:
        sample = l2_entries[0]
        print(f"    Sample: '{sample.text}' register={sample.register}")

    # Level 3 — sentence frames
    print("  Building Level 3 (sentence frames)...")
    t_l3 = time.time()
    l3_entries = tb._build_level3()
    elapsed_l3 = time.time() - t_l3
    print(f"    ✓ Level 3: {len(l3_entries):,} entries in {elapsed_l3:.1f}s")
    if l3_entries:
        sample = l3_entries[0]
        print(f"    Sample: '{sample.text}'")

    # Deduplicate
    entries = []
    seen_texts = set()
    for e in l1_entries + l2_entries + l3_entries:
        if e.text not in seen_texts:
            seen_texts.add(e.text)
            entries.append(e)

    elapsed_build = time.time() - t_build

    print(f"\n  ✓ All entries built and deduplicated")
    print(f"  Level 1            : {len(l1_entries):,}  ({elapsed_l1:.1f}s)")
    print(f"  Level 2            : {len(l2_entries):,}  ({elapsed_l2:.1f}s)")
    print(f"  Level 3            : {len(l3_entries):,}  ({elapsed_l3:.1f}s)")
    print(f"  Total (deduped)    : {len(entries):,}")
    print(f"  Total time         : {elapsed_build:.1f}s ({elapsed_build/60:.1f} min)")

    # Register distribution
    from collections import Counter
    reg_dist = Counter(e.register for e in entries)
    hedging_count = sum(1 for e in entries if e.hedging)
    print(f"\n  Register distribution:")
    for reg, cnt in reg_dist.most_common():
        print(f"    {reg:10s} : {cnt:>6,}")
    print(f"  Hedging entries    : {hedging_count:,}")

except Exception as e:
    elapsed_build = time.time() - t_build
    print(f"\n  ✗ FAILED after {elapsed_build:.1f}s: {e}")
    traceback.print_exc()
    raise

print(f"\n{'═' * 70}")
print(f"  5E COMPLETE — {elapsed_build:.1f}s ({elapsed_build/60:.1f} min)")
print(f"{'═' * 70}")

In [ ]:
# ── 5F: Save vocabulary to .npz ───────────────────────────────────
# Final step — write all ExpressionEntry objects to disk.

from src.vocabulary import VocabularyStore

print("═" * 70)
print("  STEP 5F — Save vocabulary to .npz")
print("═" * 70)

t_save = time.time()
try:
    if not entries:
        raise RuntimeError("No entries to save — check steps 5A–5E above.")

    n_entries = VocabularyStore.save(entries, VOCAB_PATH)
    elapsed_save = time.time() - t_save

    print(f"  ✓ Vocabulary saved")
    print(f"  Entries written    : {n_entries:,}")
    print(f"  File               : {VOCAB_PATH}")
    print(f"  File size          : {os.path.getsize(VOCAB_PATH) / 1024 / 1024:.2f} MB")
    print(f"  Time               : {elapsed_save:.2f}s")

except Exception as e:
    elapsed_save = time.time() - t_save
    print(f"\n  ✗ FAILED after {elapsed_save:.2f}s: {e}")
    traceback.print_exc()
    raise

# ── Full pipeline timing summary ──────────────────────────────────
total_build_time = elapsed + elapsed_place + elapsed_contrast + elapsed_calib + elapsed_build + elapsed_save
print(f"\n{'═' * 70}")
print(f"  5F COMPLETE — vocabulary saved")
print(f"{'═' * 70}")
print(f"\n  ┌─────────────────────────────────────────────────┐")
print(f"  │  FULL BUILD TIMING SUMMARY                       │")
print(f"  ├─────────────────────────────────────────────────┤")
print(f"  │  5A PMI matrix      : {elapsed:>8.1f}s  ({elapsed/60:>5.1f} min)  │")
print(f"  │  5B Word placement  : {elapsed_place:>8.1f}s  ({elapsed_place/60:>5.1f} min)  │")
print(f"  │  5C Contrast passes : {elapsed_contrast:>8.1f}s  ({elapsed_contrast/60:>5.1f} min)  │")
print(f"  │  5D Phrase radius   : {elapsed_calib:>8.1f}s  ({elapsed_calib/60:>5.1f} min)  │")
print(f"  │  5E Build entries   : {elapsed_build:>8.1f}s  ({elapsed_build/60:>5.1f} min)  │")
print(f"  │  5F Save .npz       : {elapsed_save:>8.1f}s  ({elapsed_save/60:>5.1f} min)  │")
print(f"  ├─────────────────────────────────────────────────┤")
print(f"  │  TOTAL              : {total_build_time:>8.1f}s  ({total_build_time/60:>5.1f} min)  │")
print(f"  └─────────────────────────────────────────────────┘")
print(f"\n  Words placed on M(t): {builder.n_words_placed:,}")
print(f"  Contrast judgments : {builder.n_judgments_applied:,}")
print(f"  Expression entries : {n_entries:,}")

In [ ]:
# ── Save manifold state ──────────────────────────────────────────
MANIFOLD_PATH = f"{OUTPUT_DIR}/manifold_large.npz"

n_saved = ManifoldSnapshot.save(pipeline.manifold, MANIFOLD_PATH)

print(f"✓ Manifold state saved")
print(f"  Concepts on M(t)   : {n_saved:,}")
print(f"  File               : {MANIFOLD_PATH}")
print(f"  File size          : {os.path.getsize(MANIFOLD_PATH) / 1024 / 1024:.2f} MB")
print(f"  Temperature        : {pipeline.temperature:.6f}")

## 6. Verify artifacts — load and validate

In [ ]:
# ── Verify manifold round-trip ───────────────────────────────────
info = ManifoldSnapshot.info(MANIFOLD_PATH)
print("Manifold snapshot info:")
for k, v in info.items():
    print(f"  {k:20s} : {v}")

# Spot-check: reload and compare positions
reloaded = ManifoldSnapshot.load(MANIFOLD_PATH)
labels = list(pipeline.manifold._points.keys())
max_err = 0.0
for label in labels[:100]:   # check first 100
    orig = pipeline.manifold.position(label)
    rest = reloaded.position(label)
    err = np.max(np.abs(orig - rest))
    max_err = max(max_err, err)

print(f"\n✓ Manifold round-trip verified")
print(f"  Max position error : {max_err:.2e}")
assert max_err < 1e-10, f"Round-trip error too large: {max_err}"

In [ ]:
# ── Verify vocabulary ─────────────────────────────────────────────
loaded_entries = VocabularyStore.load(VOCAB_PATH)
print(f"Vocabulary verification:")
print(f"  Entries loaded     : {len(loaded_entries):,}")

# Show a sample of entries
print(f"\n  Sample entries (first 20):")
for entry in loaded_entries[:20]:
    print(f"    [{entry.register:>7s}] [{entry.rhythm:>6s}] {entry.text[:60]}")

## 7. Evaluation — query the grown manifold

In [ ]:
# ── Load vocabulary into the renderer ─────────────────────────────
pipeline._renderer.matcher.load_vocabulary(loaded_entries)
print(f"✓ Loaded {len(loaded_entries):,} vocabulary entries into C7 renderer")

In [ ]:
# ── Run sample queries ────────────────────────────────────────────
# Pick some concepts that should be on the manifold now.

all_labels = list(pipeline.manifold._points.keys())
vocab_labels = [l for l in all_labels if l.startswith("vocab::")]
print(f"Vocabulary concepts on manifold: {len(vocab_labels):,}")
print(f"Total concepts on manifold     : {len(all_labels):,}")

# Sample some vocab words for querying
sample_words = vocab_labels[:10] if len(vocab_labels) >= 10 else vocab_labels
print(f"\nSample queries using manifold concepts:")
print("=" * 70)

for label in sample_words:
    vec = pipeline.manifold.position(label)
    result = pipeline.query(vec, label=label)
    word = label.replace("vocab::", "")
    print(f"\n  Query: {word}")
    print(f"  Steps: {result.n_steps}  |  Reason: {result.termination_reason}")
    print(f"  Confidence: {result.confidence:.3f}  |  Wave: {result.wave_confidence:.3f}")
    print(f"  Output: {result.text[:120]}")
    print("-" * 70)

In [ ]:
# ── Run evaluation suite ──────────────────────────────────────────
from src.phase5 import PipelineEvaluator

evaluator = PipelineEvaluator(pipeline)

# Pick diverse concepts for evaluation
n_eval = min(50, len(vocab_labels))
eval_labels = vocab_labels[:n_eval]
eval_vecs = [pipeline.manifold.position(l) for l in eval_labels]

print(f"Running evaluation suite with {n_eval} queries...")
t0 = time.time()
suite = evaluator.run_suite(eval_vecs, eval_labels)
elapsed = time.time() - t0

print(f"\n✓ Evaluation complete ({elapsed:.1f}s)")
print(f"  Mean coherence       : {suite.mean_coherence:.4f}")
print(f"  Novelty is decaying  : {suite.novelty_is_decaying}")
print(f"  Termination reasons  :")
for reason, count in suite.termination_distribution.items():
    print(f"    {reason:30s} : {count}")

## 8. Summary statistics

In [ ]:
# ── Final summary ─────────────────────────────────────────────────
print("=" * 70)
print("  FLOW — Kaggle Vocabulary Growth — Final Report")
print("=" * 70)
print(f"")
print(f"  Corpora")
print(f"    Wikipedia          : {source_counts.get('wikipedia', 0):,} articles")
print(f"    OpenAssistant      : {source_counts.get('openassistant', 0):,} messages")
print(f"    Alpaca (cleaned)   : {source_counts.get('alpaca', 0):,} instructions")
print(f"    Glaive Functions   : {source_counts.get('glaive_functions', 0):,} tool-call examples")
print(f"    SlimOrca           : {source_counts.get('slimorca', 0):,} reasoning chains")
print(f"    Total texts        : {len(all_texts):,}")
print(f"    Total tokens       : {builder.n_tokens_fed:,}")
print(f"")
print(f"  Vocabulary")
print(f"    Words placed on M(t): {builder.n_words_placed:,}")
print(f"    Contrast judgments   : {builder.n_judgments_applied:,}")
print(f"    Expression entries   : {n_entries:,}")
print(f"    vocabulary.npz size  : {os.path.getsize(VOCAB_PATH)/1024/1024:.2f} MB")
print(f"")
print(f"  Manifold")
print(f"    Total concepts      : {pipeline.n_concepts:,}")
print(f"    Dimension           : {pipeline.manifold.DIM}")
print(f"    Temperature         : {pipeline.temperature:.6f}")
print(f"    manifold.npz size   : {os.path.getsize(MANIFOLD_PATH)/1024/1024:.2f} MB")
print(f"")
print(f"  Evaluation")
print(f"    Mean coherence      : {suite.mean_coherence:.4f}")
print(f"    Novelty decaying    : {suite.novelty_is_decaying}")
print(f"")
print(f"  Artifacts saved to /kaggle/working/:")
print(f"    vocabulary_large.npz")
print(f"    manifold_large.npz")
print("=" * 70)
print(f"")
print(f"  Next: Upload artifacts to HuggingFace Hub or download for local use.")
print(f"  To reload locally:")
print(f"    pipeline = GEOPipeline.load('manifold_large.npz',")
print(f"                               vocabulary_path='vocabulary_large.npz')")

## 9. (Optional) Download artifacts

Kaggle saves everything in `/kaggle/working/` as output.  
After the notebook finishes, download from the Kaggle output tab:
- `vocabulary_large.npz`
- `manifold_large.npz`

Or push directly to HuggingFace Hub:

In [ ]:
# ── (Optional) Push to HuggingFace Hub ────────────────────────────
# Uncomment and set your token to push artifacts.
#
# !pip install huggingface_hub -q
# from huggingface_hub import HfApi
# api = HfApi()
#
# REPO_ID = "United-Visions/flow-geometric-manifold"
# api.create_repo(REPO_ID, exist_ok=True)
#
# api.upload_file(
#     path_or_fileobj=VOCAB_PATH,
#     path_in_repo="vocabulary_large.npz",
#     repo_id=REPO_ID,
# )
# api.upload_file(
#     path_or_fileobj=MANIFOLD_PATH,
#     path_in_repo="manifold_large.npz",
#     repo_id=REPO_ID,
# )
# print(f"✓ Artifacts pushed to https://huggingface.co/{REPO_ID}")